PDF Chatbot using LLamaIndex

In [3]:
!pip install pypdf
!pip install chromadb llama-index llama-index-vector-stores-chroma llama-index-llms-ollama llama-index-embeddings-ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [96]:
import requests
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
import threading
import os, subprocess
import time
from pathlib import Path
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from IPython.display import display, Markdown
import pypdf

In [2]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,299 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd6

In [74]:
def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [75]:
!ollama pull llama3.2
!ollama pull nomic-embed-text

In [76]:
# Configure the LLM to use Ollama with the llama3.2 model and increase the timeout
Settings.llm = Ollama(model="llama3.2", request_timeout=360.0)

# Configure the embedding model to use Ollama with the nomic-embed-text model
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

print("LlamaIndex settings updated to use Ollama for LLM (llama3.2 with increased timeout) and embedding (nomic-embed-text).")

LlamaIndex settings updated to use Ollama for LLM (llama3.2 with increased timeout) and embedding (nomic-embed-text).


In [97]:
# Get the pdf
url = "https://www.nrb.org.np/contents/uploads/2026/03/Macroeconomic-Report-February-2026.pdf"
Path("data").mkdir(exist_ok=True)
local_filename = "./data/Macroeconomic-Report-February-2026.pdf"

# Send a GET request to the URL
response = requests.get(url)

# Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Open a local file in 'wb' (write binary) mode and save the content
    with open(local_filename, "wb") as file:
        file.write(response.content)
    print("Download complete!")
else:
    print(f"Failed to download. Status code: {response.status_code}")

text_output_path = "./data/Macroeconomic-Report-February-2026.txt"

extracted_text = ""
with open(local_filename, 'rb') as file:
    reader = pypdf.PdfReader(file)
    for page_num in range(len(reader.pages)):
        page = reader.pages[page_num]
        extracted_text += page.extract_text() + "\n"

# Save the extracted text to a .txt file
with open(text_output_path, 'w', encoding='utf-8') as f:
    f.write(extracted_text)

print(f"Text extracted from PDF and saved to {text_output_path}")

# delete the pdf
if os.path.exists(local_filename):
    os.remove(local_filename)
    print("File deleted successfully.")
else:
    print("The file does not exist.")

Download complete!
Text extracted from PDF and saved to ./data/Macroeconomic-Report-February-2026.txt
File deleted successfully.


In [98]:
# Load data
documents = SimpleDirectoryReader("./data").load_data()

In [99]:
# initialize client, setting path to save data
db = chromadb.PersistentClient(path="./chroma_db")

In [100]:
# create collection
chroma_collection = db.get_or_create_collection("rag_data_collection")

In [101]:
# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [102]:
# create your index with a progress bar
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/40 [00:00<?, ?it/s]

In [115]:
from llama_index.core import set_global_handler
set_global_handler("simple")

# create a query engine and query
query_engine = index.as_query_engine()
response = query_engine.query("How were recent developments were funded?")
# print(response)
display(Markdown(response.response))

Recent data show recovery in the monetary sector and stability in the financial sector even though the overall ratio of non-performing loans in BFIs remain slightly elevated than desirable levels. Favorable inflow of remittances contributed to excess liquidity in the banking system, resulting in declining short-term interest rates in recent months.

In [116]:
# Display the source nodes for references
print("\n--- Source Nodes (References) ---")
for i, node in enumerate(response.source_nodes):
    print(f"Source Node {i+1}:\n")
    print(node.get_content().strip())
    print("\n---------------------------------")


--- Source Nodes (References) ---
Source Node 1:

The broad 
money growth's projection continued to 
be around 13.0 percent made in July 2025, 
largely supported by the growth in net 
foreign assets that largely contributed by 
the remittance inflows. Credit growth is 
expected to pick up in the next six months, 
but expected to be lower than the 12.0 
percent projection made in July 2025. 
However, the pace of credit growth depends 
upon the overall business climate, especially 
the settlement of the current political 
transition through a timely election, and 
the formation of a stable government to 
enhance the investors' confidence.  Since 
the interest rate has been historically 
low, resulting in low aggregate demand, 
increased public spending, a further boost 
in the imports, and an increase in private 
sector confidence after the formation 
of a stable government post-election. 
Moreover, growing NPL has a threefold 
impact: an increased number of blacklisted 
entrepreneurs w